# Lagos State Home Prices Prediction Model

By Jompe Emmanuel Ayomiposi, Level 3 Artificial Intelligence Project

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Import Required Libraries
# We import all the libraries needed for data manipulation, visualization,
# machine learning, and model persistence.
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd                          # Data manipulation and analysis
import numpy as np                           # Numerical computations
import matplotlib.pyplot as plt              # Plotting and visualization
import matplotlib                            # Matplotlib core
import seaborn as sns                        # Statistical data visualization
import warnings                              # Suppress unwanted warnings

from sklearn.model_selection import train_test_split    # Split dataset into train/test sets
from sklearn.preprocessing import LabelEncoder          # Encode categorical columns as integers
from sklearn.ensemble import RandomForestRegressor      # Main regression model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Evaluation metrics
import joblib                                            # Save/load trained model

warnings.filterwarnings('ignore')            # Hide deprecation warnings for cleaner output
%matplotlib inline                           # Display plots inline in the notebook

print("✅ Libraries imported successfully")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Load the Dataset
# We load the cleaned Nigerian properties CSV file and take an initial look
# at its structure, shape, and column data types.
# ─────────────────────────────────────────────────────────────────────────────

# Load the dataset from the same directory as this notebook
df = pd.read_csv('nigerian_properties_cleaned.csv')

# Display the first 5 rows to understand the structure
print("📋 First 5 rows of the dataset:")
print(df.head())

print(f"\n📐 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n📊 Column Data Types:")
print(df.dtypes)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Exploratory Data Analysis (EDA)
# We explore the dataset statistically and visually to understand distributions,
# relationships between features, and potential data quality issues.
# ─────────────────────────────────────────────────────────────────────────────

# ── 3.1 Statistical Summary ──────────────────────────────────────────────────
print("📈 Statistical Summary:")
print(df.describe())

# ── 3.2 Check for Missing Values ─────────────────────────────────────────────
print("\n🔍 Missing Values per Column:")
print(df.isnull().sum())

# ── 3.3 Unique City Count ────────────────────────────────────────────────────
print(f"\n🏙️  Unique cities in dataset: {df['city'].nunique()}")
print(df['city'].value_counts().head(10))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Visualizations (EDA Plots)
# We create visual plots to understand the distribution of property prices,
# bedroom counts, and the correlation between all numeric features.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Lagos Property Dataset — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# Plot 1: Distribution of property prices (log scale for skewed data)
axes[0, 0].hist(np.log1p(df['price']), bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribution of Property Prices (log scale)')
axes[0, 0].set_xlabel('Log(Price)')
axes[0, 0].set_ylabel('Frequency')

# Plot 2: Number of bedrooms distribution
axes[0, 1].hist(df['beds'].dropna(), bins=20, color='coral', edgecolor='white')
axes[0, 1].set_title('Distribution of Number of Bedrooms')
axes[0, 1].set_xlabel('Bedrooms')
axes[0, 1].set_ylabel('Frequency')

# Plot 3: Average price by city (top 10 cities)
top_cities = df.groupby('city')['price'].median().sort_values(ascending=False).head(10)
axes[1, 0].barh(top_cities.index, top_cities.values / 1e6, color='mediumpurple')
axes[1, 0].set_title('Median Price by City (Top 10, in Millions ₦)')
axes[1, 0].set_xlabel('Median Price (₦ Millions)')

# Plot 4: Correlation heatmap of numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols].corr()
sns.heatmap(corr, ax=axes[1, 1], annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
axes[1, 1].set_title('Correlation Heatmap')

plt.tight_layout()
plt.savefig('eda_house_price.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA plots saved as eda_house_price.png")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Data Cleaning & Preprocessing
# We clean the dataset by:
#   • Dropping the unnamed index column
#   • Removing extreme price outliers using the IQR method
#   • Filling any remaining null values
# ─────────────────────────────────────────────────────────────────────────────

# ── 5.1 Drop the unnecessary index column ────────────────────────────────────
if 'Unnamed: 0' in df.columns:
    df.drop(columns=['Unnamed: 0'], inplace=True)
    print("🗑️  Dropped 'Unnamed: 0' index column")

# ── 5.2 Remove outliers using the IQR method on the 'price' column ───────────
# Outliers can distort regression models significantly
Q1 = df['price'].quantile(0.25)   # 25th percentile
Q3 = df['price'].quantile(0.75)   # 75th percentile
IQR = Q3 - Q1                      # Interquartile Range
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

before = len(df)
df = df[(df['price'] >= lower_bound) & (df['price'] <= upper_bound)]
after = len(df)
print(f"✂️  Removed {before - after} outlier rows | Remaining: {after} rows")

# ── 5.3 Fill missing numeric values with column median ───────────────────────
# Using median (not mean) because it is less sensitive to remaining skew
for col in ['beds', 'baths', 'toilets']:
    df[col].fillna(df[col].median(), inplace=True)

print("✅ Missing values filled with column medians")
print(f"\n📋 Cleaned dataset shape: {df.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Feature Engineering
# We prepare features for model training by:
#   • Encoding the 'city' categorical column as integers (Label Encoding)
#   • Defining the feature matrix X and target vector y
# Note: 'state' column is dropped since all records are from Lagos State.
# ─────────────────────────────────────────────────────────────────────────────

# ── 6.1 Encode the 'city' column ─────────────────────────────────────────────
# Label Encoder assigns each unique city a unique integer
le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])

# Save the city encoder classes so Django can use the same mapping
city_classes = list(le.classes_)
print(f"🏙️ Encoded {len(city_classes)} unique cities")
print("Sample mapping:", {city: i for i, city in enumerate(city_classes[:5])})

# ── 6.2 Drop columns not used as features ────────────────────────────────────
# 'state' is constant (all Lagos), 'city' replaced by 'city_encoded'
df.drop(columns=['state', 'city'], inplace=True)

# ── 6.3 Define Feature Matrix (X) and Target Vector (y) ──────────────────────
X = df.drop(columns=['price'])    # All columns except the target
y = df['price']                    # Target: property price in naira

print(f"\n📐 Feature matrix X shape: {X.shape}")
print(f"📐 Target vector y shape: {y.shape}")
print(f"\n📋 Features used: {list(X.columns)}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Train / Test Split
# We split the data into 80% training and 20% testing sets.
# random_state=42 ensures reproducibility of results.
# ─────────────────────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% of data reserved for testing
    random_state=42      # Fixed seed for reproducible results
)

print("✅ Data split complete:")
print(f"   Training set  : {X_train.shape[0]} samples")
print(f"   Testing set   : {X_test.shape[0]} samples")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Model Training (Random Forest Regressor)
# We train a Random Forest Regressor — an ensemble of decision trees that
# reduces overfitting by averaging predictions from many trees.
#
# Why Random Forest for Regression?
#   • Handles non-linear relationships between features and price
#   • Robust to outliers left in data
#   • Provides built-in feature importance
#   • No need for feature scaling
# ─────────────────────────────────────────────────────────────────────────────

model = RandomForestRegressor(
    n_estimators=200,    # 200 decision trees in the forest
    max_depth=None,      # Trees grow until leaves are pure
    random_state=42,     # For reproducibility
    n_jobs=-1            # Use all available CPU cores
)

print("⏳ Training Random Forest Regressor...")
model.fit(X_train, y_train)
print("✅ Model training complete!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Model Evaluation
# We evaluate the trained model on the test set using three key metrics:
#   • MAE  (Mean Absolute Error)    — average prediction error in naira
#   • RMSE (Root Mean Squared Error) — penalises large errors more
#   • R²   (Coefficient of Det.)    — proportion of variance explained (0–1)
# ─────────────────────────────────────────────────────────────────────────────

# Generate predictions on the hold-out test set
y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("═" * 50)
print("     📊 MODEL EVALUATION RESULTS")
print("═" * 50)
print(f"  Mean Absolute Error  (MAE)  : ₦{mae:,.0f}")
print(f"  Root Mean Sq. Error  (RMSE) : ₦{rmse:,.0f}")
print(f"  R² Score             (R²)   : {r2:.4f}  ({r2*100:.2f}% variance explained)")
print("═" * 50)

# Simple interpretation
if r2 >= 0.80:
    print("\n✅ Excellent model fit! (R² ≥ 0.80)")
elif r2 >= 0.60:
    print("\n✅ Good model fit! (R² ≥ 0.60)")
elif r2 >= 0.40:
    print("\n⚠️  Moderate model fit.")
else:
    print("\n❌ Weak model fit — consider more features.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Actual vs Predicted Plot
# A scatter plot comparing actual vs predicted prices gives a visual sense
# of how closely the model tracks reality. A perfect model would produce
# all points on the diagonal line.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Performance — House Price Prediction', fontsize=14, fontweight='bold')

# Plot 1: Actual vs Predicted scatter
axes[0].scatter(y_test / 1e6, y_pred / 1e6, alpha=0.4, color='steelblue', edgecolors='white', s=30)
# Perfect prediction line
max_val = max(y_test.max(), y_pred.max()) / 1e6
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (₦ Millions)')
axes[0].set_ylabel('Predicted Price (₦ Millions)')
axes[0].set_title('Actual vs Predicted Prices')
axes[0].legend()

# Plot 2: Residuals distribution (prediction errors)
residuals = (y_test - y_pred) / 1e6
axes[1].hist(residuals, bins=40, color='coral', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual (₦ Millions)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prediction Errors (Residuals)')

plt.tight_layout()
plt.savefig('model_performance_house_price.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Performance plots saved")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — Feature Importance
# Random Forests can rank how much each feature contributed to predictions.
# This helps interpret the model and understand which property attributes
# most strongly influence price.
# ─────────────────────────────────────────────────────────────────────────────

# Extract feature importances from the trained model
importances = model.feature_importances_
feature_names = X.columns.tolist()

# Sort features by importance (descending)
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
colors = ['steelblue' if i == 0 else 'lightsteelblue' for i in range(len(feature_names))]
plt.bar(
    [feature_names[i] for i in sorted_idx],
    [importances[i] for i in sorted_idx],
    color=colors
)
plt.title('Feature Importance — House Price Prediction Model', fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('feature_importance_house_price.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Feature Importance Ranking:")
for i in sorted_idx:
    print(f"   {feature_names[i]:<15} : {importances[i]:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Save the Trained Model
# We save the trained model and the label encoder to disk using joblib.
# These saved files will be loaded by the Django web interface for
# real-time predictions without retraining.
# ─────────────────────────────────────────────────────────────────────────────

import os

# Save the trained regression model
model_path = 'house_price_model.pkl'
joblib.dump(model, model_path)
print(f"✅ Model saved to: {os.path.abspath(model_path)}")

# Save the city label encoder so Django can encode user input the same way
encoder_path = 'city_label_encoder.pkl'
joblib.dump(le, encoder_path)
print(f"✅ City encoder saved to: {os.path.abspath(encoder_path)}")

# Save city names list for use in the Django form dropdown
city_list_path = 'city_classes.pkl'
joblib.dump(city_classes, city_list_path)
print(f"✅ City list saved to: {os.path.abspath(city_list_path)}")

print("\n🎉 House Price Prediction Model is ready for deployment!")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Quick Prediction Demo
# Test the saved model with a sample property to verify it works correctly.
# This simulates what the Django interface will do when a user submits a form.
# ─────────────────────────────────────────────────────────────────────────────

# Load the saved model from disk
loaded_model   = joblib.load('house_price_model.pkl')
loaded_encoder = joblib.load('city_label_encoder.pkl')

# ── Sample property to predict ────────────────────────────────────────────────
sample_city = 'ajah'

# Encode the city using the saved encoder
sample_city_encoded = loaded_encoder.transform([sample_city])[0]

# Create a sample property as a DataFrame (must match training feature order)
sample = pd.DataFrame({
    'beds'         : [4],
    'baths'        : [3],
    'toilets'      : [4],
    'city_encoded' : [sample_city_encoded]
})

# Predict the price
predicted_price = loaded_model.predict(sample)[0]

print("🏠 Sample Prediction Demo")
print(f"   City    : {sample_city}")
print(f"   Beds    : 4  |  Baths : 3  |  Toilets : 4")
print(f"   Predicted Price: ₦{predicted_price:,.0f}")
print(f"   (~₦{predicted_price/1e6:.1f} Million)")